In [38]:
# Import libraries and load datasets
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import sqlite3
from wordcloud import WordCloud
from datetime import datetime #only need date, not time

df_choc = pd.read_csv('../Data/chocolate_bar_ratings_2022_cleaned.csv')
df_coffee = pd.read_csv('../Data/simplified_coffee_cleaned.csv')

In [39]:
df_choc.head()

,REF,Company,Company Location,Review Date,Bean Origin,Bar Name,Cocoa %,Ingredients,Characteristics,Rating
0,2454,5150,U.S.A.,2019,Tanzania,"Kokoa Kamili, batch 1",76.0,"3- B,S,C","rich cocoa, fatty, bready",3.25
1,2454,5150,U.S.A.,2019,Madagascar,"Bejofo Estate, batch 1",76.0,"3- B,S,C","cocoa, blackberry, full body",3.75
2,2458,5150,U.S.A.,2019,Dominican Republic,"Zorzal, batch 1",76.0,"3- B,S,C","cocoa, vegetal, savory",3.50
3,2542,5150,U.S.A.,2021,Fiji,"Matasawalevu, batch 1",68.0,"3- B,S,C","chewy, off, rubbery",3.00
4,2542,5150,U.S.A.,2021,India,"Anamalai, batch 1",68.0,"3- B,S,C","milk brownie, macadamia,chewy",3.50


In [40]:
df_coffee.head()

,name,roaster,roast,loc_country,origin,100g_USD,rating,review_date,review
0,Ethiopia Shakiso Mormora,Revel Coffee,Medium-Light,United States,Ethiopia,4.70,92,November 2017,"Crisply sweet, cocoa-toned. Lemon blossom, roa..."
1,Ethiopia Suke Quto,Roast House,Medium-Light,United States,Ethiopia,4.19,92,November 2017,"Delicate, sweetly spice-toned. Pink peppercorn..."
2,Ethiopia Gedeb Halo Beriti,Big Creek Coffee Roasters,Medium,United States,Ethiopia,4.85,94,November 2017,"Deeply sweet, subtly pungent. Honey, pear, tan..."
3,Ethiopia Kayon Mountain,Red Rooster Coffee Roaster,Light,United States,Ethiopia,5.14,93,November 2017,"Delicate, richly and sweetly tart. Dried hibis..."
4,Ethiopia Gelgelu Natural Organic,Willoughby's Coffee & Tea,Medium-Light,United States,Ethiopia,3.97,93,November 2017,"High-toned, floral. Dried apricot, magnolia, a..."


I decided to merge these two DataFrames on the manufacturer/roaster location, bean origin columns, review, and company/roaster columns. To do this, I will need to rename these four columns in the chocolate dataset. I'll also need to rename the roaster and name column in the coffee dataset. For uniformity, I will make all other columns in the chocolate dataset lower case. Finally, I'll drop the REF column in the chocolate dataset, because it is not relevant to this analysis.

In [41]:
df_choc = df_choc.drop(columns='REF') #assigning to a variable so it doesn't change the original dataset.

In [42]:
df_choc.columns

Index(['Company', 'Company Location', 'Review Date', 'Bean Origin', 'Bar Name',
       'Cocoa %', 'Ingredients', 'Characteristics', 'Rating'],
      dtype='str')

In [43]:
df_coffee.columns

Index(['name', 'roaster', 'roast', 'loc_country', 'origin', '100g_USD',
       'rating', 'review_date', 'review'],
      dtype='str')

In [49]:
# rename columns in chocolate dataset
df_choc = df_choc.rename(columns={'Company': 'company', 'Company Location': 'loc_country', 'Review Date': 'review_date', 'Bean Origin': 'origin', 'Bar Name': 'bar_name', 'Cocoa %': 'cocoa_%', 'Ingredients': 'ingredients', 'Characteristics': 'review', 'Rating': 'rating'})
df_choc.columns

Index(['company', 'loc_country', 'review_date', 'origin', 'bar_name',
       'cocoa_%', 'ingredients', 'review', 'rating'],
      dtype='str')

In [50]:
# rename columns in coffee dataset
df_coffee = df_coffee.rename(columns={'name': 'coffee_name', 'roaster': 'company'})

Now I will attempt to merge the two DataFrames. According to Pandas documentation (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.merge.html), a left join uses only the keys from the left frame, and an outer join uses union of keys from both frames. I believe that I want an outer join because I want it to include all of the columns. If that doesn't look right, then I'll try a left join.

In [34]:
# I tried to run df_combined = df_choc.merge(df_coffee, how = 'outer') and received an error message.

The error message stated that I'm trying to merge a string value and an integer for review date. The chocolate dataset only contains the year, whereas the coffee dataset contains the month and year. I needed to get rid of the month and then change the year to an integer in the coffee dataset.      

Google search revealed that I would need to import date from the datetime module. No PIP install is needed, as datetime is part of Python's standard library.    

I tried unsuccessfully to create a function here, and the more I researched, I realized that a function is not needed.  If I used AI to help me create a function, I was concerned that I would not be able to explain it, so I did not create one. 

Hypothetically, if I were to create a function, here is what it would need to do in order to change a string object like 'November 2017' to an integer like '2017':
1. Iterate over the review_date column in df_coffee: column df_coffee['review_date']
1. Extract the year from the month-year string
2. Convert the year string to an integer # num = int(s)
3. Handle any exceptions and print out an "invalid" message in case there's a typo - can only do this if it's not a string, though.
4. Return an integer (place it in the column)
5. Assign the column name to a variable coffee_date = df_coffee['review_date']
6. For loop (for month_year in coffee_date)
7. %Y would be the 4-digit year
8. Don't need to convert the column (series) into a list
9. Handle something that's not a month-year string with exception error, if/else



I don't want to drop the review_date column in case I need it later. The above error message had also stated, "If you wish to proceed you should use pd.concat." When researching the concatenate function, it appears that it will create a dataframe with duplicate columns. I think the duplicate columns will be ok because I will be creating ERD tables and a database from it.  So, concat it is.

In [51]:
df_combined = pd.concat([df_choc, df_coffee], axis=1) #on the columns
df_combined.head()

,company,loc_country,review_date,origin,bar_name,cocoa_%,ingredients,review,rating,coffee_name,company,roast,loc_country,origin,100g_USD,rating,review_date,review
0,5150,U.S.A.,2019,Tanzania,"Kokoa Kamili, batch 1",76.0,"3- B,S,C","rich cocoa, fatty, bready",3.25,Ethiopia Shakiso Mormora,Revel Coffee,Medium-Light,United States,Ethiopia,4.70,92.0,November 2017,"Crisply sweet, cocoa-toned. Lemon blossom, roa..."
1,5150,U.S.A.,2019,Madagascar,"Bejofo Estate, batch 1",76.0,"3- B,S,C","cocoa, blackberry, full body",3.75,Ethiopia Suke Quto,Roast House,Medium-Light,United States,Ethiopia,4.19,92.0,November 2017,"Delicate, sweetly spice-toned. Pink peppercorn..."
2,5150,U.S.A.,2019,Dominican Republic,"Zorzal, batch 1",76.0,"3- B,S,C","cocoa, vegetal, savory",3.50,Ethiopia Gedeb Halo Beriti,Big Creek Coffee Roasters,Medium,United States,Ethiopia,4.85,94.0,November 2017,"Deeply sweet, subtly pungent. Honey, pear, tan..."
3,5150,U.S.A.,2021,Fiji,"Matasawalevu, batch 1",68.0,"3- B,S,C","chewy, off, rubbery",3.00,Ethiopia Kayon Mountain,Red Rooster Coffee Roaster,Light,United States,Ethiopia,5.14,93.0,November 2017,"Delicate, richly and sweetly tart. Dried hibis..."
4,5150,U.S.A.,2021,India,"Anamalai, batch 1",68.0,"3- B,S,C","milk brownie, macadamia,chewy",3.50,Ethiopia Gelgelu Natural Organic,Willoughby's Coffee & Tea,Medium-Light,United States,Ethiopia,3.97,93.0,November 2017,"High-toned, floral. Dried apricot, magnolia, a..."


In [ ]:
# Get a list of all of the column names

In [52]:
df_combined.columns

Index(['company', 'loc_country', 'review_date', 'origin', 'bar_name',
       'cocoa_%', 'ingredients', 'review', 'rating', 'coffee_name', 'company',
       'roast', 'loc_country', 'origin', '100g_USD', 'rating', 'review_date',
       'review'],
      dtype='str')

Since there are duplicate columns in one DataFrame, I have decided that I need to rename them again.  